# 1회차 · 한 대를 띄우는 피드백

오늘 끝에는 내가 쓴 PID가 드론을 0.5 m에서 1.5 m로 올리고 그 자리에 세운다.

**진행 방식**: 설명을 듣다가 `✍️` 표시가 있는 셀에서 같이 코드를 치고, 바로 실행(`Shift+Enter`)해서 그래프로 확인한다.
빈칸은 `___` 이다. 막히면 옆 사람 화면을 보거나 손을 들자. 다음 단계 셀에는 앞 단계 코드가 미리 채워져 있으니
한 번 놓쳐도 다음 단계부터 다시 따라오면 된다.

| 단계 | 내용 |
|---|---|
| 0 | 준비: 시뮬레이터 켜기 |
| 1 | 호버 추력: 가만히 떠 있으려면 |
| 2 | 피드백과 P: 어긋난 만큼 고친다 |
| 3 | D: 브레이크 |
| 4 | I: 모자란 힘 채우기 |
| 5 | 멀리 있을 땐 적분 쉬기 |
| 6 | 미션: 내 게인으로 두 기체 통과 |
| 7 | 실시간 비행: 날고 있는 동안 값 바꾸기, PlotJuggler |

## 0. 준비

아래 셀을 실행한다. 처음 한 번은 물리 엔진을 준비하느라 몇 초 걸린다.
오늘 기체는 손바닥만 한 Crazyflie급 드론이고, 위아래(z)로만 움직인다. 기울지 않게 잡는 일은 배경 코드가 맡는다.

In [ ]:
from background import *

warmup()
drone_info()

## 1. 호버 추력: 가만히 떠 있으려면

드론에 걸리는 위아래 힘은 둘뿐이다. 모터가 미는 **추력 T**와 **무게 mg**.

$$m\ddot z = T - mg \quad\Rightarrow\quad \text{떠 있으려면 } T = mg$$

✍️ 떠 있는 데 필요한 추력을 돌려주는 함수를 완성하자. 9.81 대신 인자 `g`를 쓴다.

In [ ]:
def hover_thrust(mass, g=G):
    return ___  # ✍️


fly_open_loop(hover_thrust)

**보고 생각하기**
- 파란 선(질량이 정확할 때)은 그대로 떠 있다.
- 실제 기체가 10% 무거우면? 10% 가벼우면?
- 배터리, 마커, 바람 때문에 실제 기체는 늘 조금씩 다르다. 처음 정한 힘만 믿고 미는 것(열린 루프)으로는 못 버틴다.
  **높이를 보면서 고쳐야 한다.** 이것이 피드백이다.

## 2. 피드백과 P 제어: 어긋난 만큼 고친다

1초에 100번 이렇게 한다.

1. **본다**: 지금 높이 `z`
2. **비교한다**: 오차 `error = 목표 - 지금` (위로 가야 하면 +)
3. **고친다**: 오차를 보고 원하는 가속도 `acc`를 정해서, 호버 추력에 더한다. `T = hover + m·acc`

가장 단순한 규칙은 오차에 비례해서 미는 것, **P 제어**다. `acc = kp × error`.
용수철과 같다. 멀리 늘일수록 세게 당긴다. `kp`는 용수철의 단단함.

✍️ 세 곳을 채운다: P 제어의 출력, 오차, 그리고 추력.
추력은 $T = \hat m\,(g + u) = \hat m g + \hat m u$ 를 코드로 옮긴 것이다.
$\hat m g$는 1단계에서 만든 호버 추력(컨트롤러가 아는 질량은 `MASS`), $u$는 바로 위 줄에서 받은 원하는 가속도 `acc`.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        return ___  # ✍️


def altitude_thrust(pid, z_target, z, dt):
    error = ___  # ✍️
    acc = pid.update(error, dt)         # 원하는 위쪽 가속도 [m/s²]
    return ___  # ✍️ 추력 T [N] = 호버 추력 + 질량 × 원하는 가속도


step_test(PID(kp=8), altitude_thrust)

**보고 생각하기**: 4초에 목표가 1.5 m로 바뀐다.
- 목표에 멈추나? 브레이크가 없는 용수철처럼 출렁인다.
- 출렁임의 가운데가 목표보다 아래다. 무거운 10%를 버티려면 오차가 남아 있어야 해서 **처진다**.

아래 셀에서 `kp`를 바꿔 보자. 약하면? 세면?

In [ ]:
compare({"kp = 2": PID(kp=2), "kp = 8": PID(kp=8), "kp = 20": PID(kp=20)}, altitude_thrust)

## 3. D 제어: 브레이크

빨간불 앞에서 미리 브레이크를 밟듯이, **오차가 빨리 줄어들고 있으면 미리 덜 민다.**
오차가 변하는 빠르기는 `(지금 오차 - 직전 오차) / dt`.

$$acc = k_p\,e + k_d\,\frac{e - e_{\text{직전}}}{dt}$$

처음 부를 때는 직전 오차가 없으니 0으로 둔다.

✍️ 미분(빠르기)을 계산하고, 출력에 D를 더한다.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        if self.prev_error is None:
            derivative = ___  # ✍️ 처음 부를 때
        else:
            derivative = ___  # ✍️ (지금 오차 - 직전 오차) / dt
        self.prev_error = error
        return ___  # ✍️


compare({"P (kp 8)": PID(kp=8), "PD (kp 8, kd 4)": PID(kp=8, kd=4)}, altitude_thrust)

**보고 생각하기**
- 출렁임이 잦아들었다. 그런데 무거운 기체에서는 목표보다 약 12 cm 아래에 서 있다.
  브레이크는 움직임을 막을 뿐, 모자란 힘을 채워 주지는 못한다.
- `kd`를 2, 4, 8로 바꿔 보자. 너무 작으면? 너무 크면?

In [ ]:
compare({"kd = 2": PID(kp=8, kd=2), "kd = 4": PID(kp=8, kd=4), "kd = 8": PID(kp=8, kd=8)}, altitude_thrust)

## 4. I 제어: 모자란 힘 채우기

**새지 않는 물통**을 생각하자. 드론이 목표보다 아래에 있는 동안, 물통에 물을 **계속 조금씩 붓는다.**
물통의 물이 많을수록 드론을 더 세게 받쳐 준다. 물통은 새지 않으니, 목표에 도착해 오차가 0이 되면 붓기만 멈추고
이미 부은 물은 그대로 남아서 모자란 무게를 계속 받친다.
코드로는 오차를 시간에 따라 계속 더해 두는 것(적분)이다. 처짐이 남아 있는 동안 그 값이 자라서 결국 모자란 힘만큼 채운다.

$$acc = k_p\,e + k_i \textstyle\sum e\,dt + k_d\,\dot e$$

✍️ 적분에 `error × dt`를 더하고, 출력에 I를 더한다.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        self.integral += ___  # ✍️
        if self.prev_error is None:
            derivative = 0.0
        else:
            derivative = (error - self.prev_error) / dt
        self.prev_error = error
        return ___  # ✍️


compare({"PD": PID(kp=8, kd=4), "PID (ki 8)": PID(kp=8, ki=8, kd=4)}, altitude_thrust)

**보고 생각하기**
- 처짐은 사라졌다. 그런데 이번에는 1.5 m를 크게 지나친다. 왜일까?
- 1 m를 올라가는 동안 오차가 크게 남아 있었고, 그동안 적분이 잔뜩 쌓였다(물을 너무 많이 부었다).
  도착한 뒤에도 쌓인 만큼 더 밀어서 지나친다. 샤워기를 끝까지 돌렸다가 데이는 것과 같다.

## 5. 멀리 있을 땐 적분 쉬기

고치는 방법은 간단하다. **목표 근처(오차가 `i_band`보다 작을 때)에서만 적분한다.**
멀리 있을 때는 P와 D가 충분히 세게 밀어 주니까, I는 마지막 몇 cm만 맡으면 된다.

✍️ 적분하는 줄 앞에 조건을 단다. `i_band`가 `None`이면 늘 적분한다.

In [ ]:
class PID:
    def __init__(self, kp, ki=0.0, kd=0.0, i_band=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.i_band = i_band
        self.integral = 0.0
        self.prev_error = None

    def update(self, error, dt):
        if ___:  # ✍️ 오차가 i_band보다 작을 때만 (i_band가 None이면 늘)
            self.integral += error * dt
        if self.prev_error is None:
            derivative = 0.0
        else:
            derivative = (error - self.prev_error) / dt
        self.prev_error = error
        return self.kp * error + self.ki * self.integral + self.kd * derivative


compare({"PID": PID(kp=8, ki=8, kd=4), "PID + 가까울 때만 적분": PID(kp=8, ki=8, kd=4, i_band=0.15)}, altitude_thrust)

## 6. 미션: 내 게인으로 두 기체 모두 통과

기준은 세 가지다. **오버슈트 < 10%**, **2% 정착 < 2초**, **마지막 1초 오차 < 2 cm**.
질량이 맞는 기체와 10% 무거운 기체, 둘 다 통과해야 한다 (여섯 칸 모두 초록).

순서: `kp`를 올려 빨리 가게 → `kd`로 출렁임 잡기 → `ki`로 남은 처짐 없애기.

✍️ 숫자를 바꿔 가며 다시 실행해 보자.

In [ ]:
MY_GAINS = dict(kp=___, ki=___, kd=___, i_band=0.15)  # ✍️

mission(PID, MY_GAINS, altitude_thrust)

슬라이더로 바꾸면 놓는 순간 바로 다시 날린다.

In [ ]:
tune(PID, altitude_thrust)

## 7. 실시간으로 보기: 날고 있는 동안 값 바꾸기

지금까지는 10초 비행을 한꺼번에 계산하고 그래프를 그렸다. 실제 드론은 지금 이 순간 날고 있고,
우리는 그걸 보면서 값을 바꾼다. 아래 셀은 드론을 **1배속으로** 계속 띄운다 (최대 2분, ■ 멈춤으로 끝).

날고 있는 동안 해 볼 것:
- **목표 높이**를 올렸다 내리기: 계단 시험을 손으로 하는 셈이다
- **kd**를 0 가까이: 출렁임이 바로 커진다. 다시 올리면 가라앉는다
- **ki**를 0으로: 10% 무거운 기체가 목표보다 조금 아래에서 멈춘다

가운데 그래프는 PID 출력을 세 항으로 나눈 것이다. 목표를 바꾸는 순간 D가 크게 튀는 것도 보인다.
`window=True`를 넣으면 3D 창도 같이 뜬다.

In [ ]:
live(PID, altitude_thrust, **MY_GAINS)

### PlotJuggler로 보기 (선택)

실제 드론 팀이 비행 기록을 볼 때 쓰는 그래프 도구다. 왼쪽 목록의 값을 그래프 칸으로 끌어 놓기만 하면 되고,
멈춘 뒤에는 되감아 볼 수도 있다. 아래 셀로 창을 열고 **Yes → OK**를 누른 다음, 위의 실시간 비행 셀을 다시 실행한다.
오늘 그래프 배치(높이, P·I·D, 추력)가 미리 들어 있다.

In [ ]:
plotjuggler("week01")

## 정리

| 오늘 쓴 것 | 한 줄 요약 |
|---|---|
| `hover_thrust` | 떠 있으려면 T = mg. 이것만으로는 못 버틴다 |
| P | 어긋난 만큼 민다. 출렁이고 처진다 |
| D | 빨리 다가가면 미리 덜 민다. 출렁임을 잡는다 |
| I | 남은 오차를 쌓아서 채운다. 처짐을 없앤다 |
| 가까울 때만 적분 | I가 너무 쌓여 지나치는 것을 막는다 |
| 실시간 비행 | 날면서 게인을 바꾸면 그래프가 바로 달라진다 |

다음 시간에는 같은 PID를 옆(x, y)으로 쓴다. 드론은 기울여야 옆으로 간다.